In [1]:
!pip install deap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.4/111.4 kB 1.4 MB/s eta 0:00:00a 0:00:01


In [8]:
from deap import base, creator, tools, algorithms
import random
import numpy as np

# Define the problem as a minimization problem
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

# Define the genetic algorithm parameters
toolbox = base.Toolbox()
toolbox.register("attr_float", random.uniform, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_float, n=10)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# Define the evaluation function using a neural network model
def evaluate(individual):
    # Placeholder for neural network prediction
    # Replace with actual neural network model prediction
    return (np.sum(individual),)

toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxBlend, alpha=0.5)
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=0.1, indpb=0.2)
toolbox.register("select", tools.selTournament, tournsize=3)

# Genetic algorithm execution
population = toolbox.population(n=50)
ngen, cxpb, mutpb = 40, 0.7, 0.2
algorithms.eaSimple(population, toolbox, cxpb, mutpb, ngen, verbose=True)


gen	nevals
0  	50    
1  	38    
2  	38    
3  	33    
4  	39    
5  	39    
6  	35    
7  	32    
8  	41    
9  	44    
10 	44    
11 	44    
12 	35    
13 	36    
14 	39    
15 	33    
16 	34    
17 	41    
18 	40    
19 	37    
20 	34    
21 	39    
22 	39    
23 	36    
24 	38    
25 	42    
26 	34    
27 	35    
28 	36    
29 	37    
30 	39    
31 	40    
32 	35    
33 	41    
34 	34    
35 	34    
36 	38    
37 	37    
38 	37    
39 	41    
40 	37    


([[-0.4377105556633748,
   -0.7361697562495588,
   -2.060129330130544,
   -1.0663455365355317,
   -7.578557208283274,
   -1.341968382265018,
   -1.5292390547583952,
   -2.249920099932195,
   -0.972542597440161,
   -1.3545527803270927],
  [-0.45003578885902984,
   -0.637433442828494,
   -1.8188011063071363,
   -1.0683031131628322,
   -7.574495282032359,
   -0.8036499390245389,
   -1.6323133624622903,
   -2.2822682517897164,
   -0.9342865393845347,
   -1.819880252783212],
  [-0.4316277579831668,
   -0.66946163811219,
   -1.8968007670444194,
   -1.0804555641365619,
   -7.531972762750938,
   -0.8434738795043573,
   -1.5344246962659331,
   -2.288553053709962,
   -0.9342273614488856,
   -1.6530253575855551],
  [-0.451370659465486,
   -0.6892205975126573,
   -1.6758727435455978,
   -1.1603125732885102,
   -8.093050023532607,
   -0.6048166659731521,
   -1.517758719256475,
   -2.273232416107302,
   -0.9569869245952489,
   -1.630664301740891],
  [-0.4521376059239439,
   -0.4381610711266133,
   -

In [14]:
from deap import base, creator, tools, algorithms
import random
import numpy as np
import warnings
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Clear any existing DEAP definitions to avoid conflicts
try:
    del creator.FitnessMin
    del creator.Individual
except:
    pass

# Define the problem as a minimization problem
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

# Define the genetic algorithm parameters
toolbox = base.Toolbox()
toolbox.register("attr_float", random.uniform, 0, 1)  # Random float values in range [0, 1]
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_float, n=3)  # Individual with 3 attributes
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# Placeholder for neural network model
X = np.random.rand(100, 5)  # Example features (replace with actual features)
y = np.random.rand(100)  # Example target (replace with actual target)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the fitness function for the genetic algorithm
def evaluate(individual):
    # Unpack genetic algorithm parameters - scale to appropriate ranges
    population_size = int(individual[0] * 190 + 10)  # Scale to range [10, 200]
    crossover_rate = individual[1]  # Already in range [0, 1]
    mutation_rate = individual[2]  # Already in range [0, 1]
    
    try:
        # Initialize and tune the neural network model parameters
        nn_model = MLPRegressor(
            hidden_layer_sizes=(population_size, 50), 
            activation='relu', 
            solver='adam', 
            random_state=42,
            max_iter=100  # Limit iterations to make evaluation faster
        )
        
        # Train the neural network model
        nn_model.fit(X_train, y_train)
        
        # Evaluate the model on the test set
        y_pred = nn_model.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)  # Fitness function is based on MSE
        
        return (mse,)
    except Exception as e:
        # Return a high error value if something fails
        print(f"Error during evaluation: {e}")
        return (float('inf'),)

toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxBlend, alpha=0.5)  # Crossover method (blend)
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=0.1, indpb=0.2)  # Mutation method (Gaussian)
toolbox.register("select", tools.selTournament, tournsize=3)  # Selection method (Tournament)

# Initialize population
population = toolbox.population(n=20)  # Reduced population size for faster execution
ngen, cxpb, mutpb = 10, 0.7, 0.2  # Reduced generations for faster execution

# Run the genetic algorithm
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("min", np.min)
stats.register("max", np.max)

# Execute the algorithm and unpack the results
result, logbook = algorithms.eaSimple(population, toolbox, cxpb, mutpb, ngen, 
                                     stats=stats, verbose=True)

# Extract the best individual (optimized GA parameters)
best_individual = tools.selBest(population, 1)[0]
pop_size = int(best_individual[0] * 190 + 10)
cx_rate = best_individual[1]
mut_rate = best_individual[2]

print("\nBest GA Parameters:")
print(f"Population Size: {pop_size}")
print(f"Crossover Rate: {cx_rate:.2f}")
print(f"Mutation Rate: {mut_rate:.2f}")

gen	nevals	avg      	min      	max     
0  	20    	0.0956254	0.0843173	0.113058
1  	16    	0.0938072	0.0843173	0.106804
2  	16    	0.0977258	0.0843173	0.129182
3  	11    	0.0919439	0.0843173	0.107295
4  	13    	0.0898974	0.0843173	0.102572
5  	18    	0.0843173	0.0843173	0.0843173
6  	12    	0.0843173	0.0843173	0.0843173
7  	17    	0.0843173	0.0843173	0.0843173
8  	12    	0.0843173	0.0843173	0.0843173
9  	14    	0.0843173	0.0843173	0.0843173
10 	15    	0.0850127	0.0843173	0.098226 

Best GA Parameters:
Population Size: 148
Crossover Rate: 0.49
Mutation Rate: 0.04
